# Tour Planning Script - Status Tracker

This notebook processes delivery orders and creates optimized tour plans using the HERE Tour Planning API.

### Process Overview

The tour planning process consists of the following steps:

1. **Initialization** - Load modules and set up paths
2. **File Upload** - Upload and validate input Excel file
3. **Data Cleaning** - Clean, normalize, and filter order data
4. **Geocoding** - Geocode pickup and delivery addresses
5. **VRP Planning** - Create and execute vehicle routing optimization
6. **Results Merging** - Merge VRP results with order data
7. **Export** - Generate CSV files for bexOS import
8. **Visualization** - Create interactive map of tours

### Status Tracking

This notebook includes comprehensive status tracking. After each step, check the status output for:
- ✅ **Success** - Step completed successfully
- ⚠️ **Warnings** - Issues that may affect results but don't stop processing
- ❌ **Errors** - Critical issues that need to be fixed

### Common Issues

Most errors come from **wrong input files**. Common problems:
- Missing required columns
- Wrong sheet selected
- Incorrect date format
- Empty or invalid data

The validation system will help identify these issues early.
---
Results are saved to: [this folder](https://drive.google.com/drive/folders/1UHngYb9uXdRfO_0N002zc2Rz0xnk3xbX?usp=drive_link)

In [3]:
# @title Step 1: Initialization and Setup

from google.colab import drive
import sys, os, importlib

# 0. Initial Loading of parameters

# From colab secrets
from google.colab import userdata
api_key = userdata.get('apiKey_HERE')

# Define Planning Date
base_date_str = "2026-02-12" # @param {"type":"date"}

# Specify client name
client_name = "Wigger" # @param ["Stark","Kemmler","laminatDepot_MultiBranch","Wigger","Obi_Buchholz","default"]

# Specify product category
product_category = "Ohne Modifikation" # @param ["Ohne Modifikation","bex Kurier","bex Tour"]

# Set paths based on client
if client_name == "Stark":
    depots_path = '/content/drive/MyDrive/Colab Notebooks/stark/depots/Depots_geocoded.xlsx'
    output_folder_path = '/content/drive/MyDrive/Colab Notebooks/stark/outputs/' + base_date_str
    sheet_name = 'Sheet1'
elif client_name == "Kemmler":
    depots_path = '/content/drive/MyDrive/Colab Notebooks/kemmler/depots/Depots_geocoded.xlsx'
    output_folder_path = '/content/drive/MyDrive/Colab Notebooks/kemmler/outputs/' + base_date_str
    sheet_name = 'Touren - BEX'
elif client_name == "laminatDepot_MultiBranch":
    depots_path = '/content/drive/MyDrive/Colab Notebooks/laminatdepot/depots/Depots_geocoded.xlsx'
    output_folder_path = '/content/drive/MyDrive/Colab Notebooks/laminatdepot/outputs/' + base_date_str
    sheet_name = 'input'
elif client_name == "Wigger":
    depots_path = '/content/drive/MyDrive/Colab Notebooks/Wigger/depots/Depots_geocoded.xlsx'
    output_folder_path = '/content/drive/MyDrive/Colab Notebooks/Wigger/outputs/' + base_date_str
    sheet_name = 'Touren - BEX'
elif client_name == "Obi_Buchholz":
    depots_path = '/content/drive/MyDrive/Colab Notebooks/Obi_Buchholz/depots/Depots_geocoded.xlsx'
    output_folder_path = '/content/drive/MyDrive/Colab Notebooks/Obi_Buchholz/outputs/' + base_date_str
    sheet_name = 'Touren - BEX'
else:
    depots_path = '/content/drive/MyDrive/Colab Notebooks/generic_tourplanning/depots/Depots_geocoded.xlsx'
    output_folder_path = '/content/drive/MyDrive/Colab Notebooks/generic_tourplanning/outputs/' + base_date_str
    sheet_name = 'Sheet1'

# 1. Mount Drive for Drive access
# Check if Drive is already mounted
if os.path.exists('/content/drive/MyDrive'):
    print("✅ Google Drive is already mounted")
    drive_mount_message = "Google Drive was already mounted"
else:
    try:
        drive.mount('/content/drive')  # can skip if only widget upload needed
        print("✅ Google Drive mounted successfully")
        drive_mount_message = "Google Drive mounted successfully"
    except Exception as e:
        # If mount fails because it's already mounted, use existing mount
        if "already contain files" in str(e) or "already mounted" in str(e).lower():
            print("⚠️  Drive appears to be mounted. Using existing mount...")
            drive_mount_message = "Google Drive was already mounted (using existing mount)"
        else:
            print(f"❌ Failed to mount Drive: {str(e)}")
            raise

# 2. Add module folder to path
module_path = '/content/drive/MyDrive/Colab Notebooks/tourplanning_modules'
if module_path not in sys.path:
    sys.path.insert(0, module_path)

# 3. Import modules
module_loading_error = None
try:
    import module_upload, module_clean, module_geocoding, module_vrp, module_results, module_map, module_column_mapping, module_export, module_client_configuration
    import module_input_validation, module_tour_summary, module_status_tracker

    # 4. Reload modules after any change
    importlib.reload(module_upload)
    importlib.reload(module_clean)
    importlib.reload(module_geocoding)
    importlib.reload(module_vrp)
    importlib.reload(module_results)
    importlib.reload(module_map)
    importlib.reload(module_column_mapping)
    importlib.reload(module_export)
    importlib.reload(module_client_configuration)
    importlib.reload(module_input_validation)
    importlib.reload(module_tour_summary)
    importlib.reload(module_status_tracker)

    module_loading_success = True
    print("✅ All modules loaded successfully")
except Exception as e:
    module_loading_error = str(e)
    print(f"❌ Failed to import modules: {module_loading_error}")
    print(f"   Module path: {module_path}")
    print(f"   Python path entries: {sys.path[:3]}")
    print(f"   Please check that all module files exist in the tourplanning_modules folder")
    module_loading_success = False
    # Try to import at least the status tracker for error reporting
    try:
        import module_status_tracker
        importlib.reload(module_status_tracker)
    except:
        pass

# 5. Initialize status tracker (after modules are loaded)
try:
    from module_status_tracker import get_tracker, reset_tracker
    tracker = get_tracker()
    reset_tracker()  # Reset for fresh run
    tracker.start()

    tracker.add_step("Drive Mount", "success", drive_mount_message)
    if module_loading_success:
        tracker.add_step("Module Loading", "success", "All modules loaded and reloaded")
    else:
        error_msg = module_loading_error if module_loading_error else "Unknown error"
        tracker.add_step("Module Loading", "error", f"Failed to load modules: {error_msg}",
                        {"hint": "Check that all module files exist in the tourplanning_modules folder"})
except Exception as e:
    print(f"⚠️  Could not initialize status tracker: {str(e)}")
    # Create a dummy tracker object to prevent NameError
    class DummyTracker:
        def add_step(self, *args, **kwargs): pass
        def end(self): pass
        def print_summary(self): print("Status tracker not available")
        def get_steps_dataframe(self): return None
        def get_summary(self): return {"status": "unknown"}
    tracker = DummyTracker()

# 6. Load API key and validate
try:
    if not api_key:
        raise ValueError("API key not found in Colab secrets")
    tracker.add_step("API Key", "success", "API key loaded from Colab secrets")
except Exception as e:
    tracker.add_step("API Key", "error", f"Failed to load API key: {str(e)}",
                    {"hint": "Make sure 'apiKey_HERE' is set in Colab Secrets (🔑 icon)"})
    raise

# 7. Create output folder if it doesn't exist
os.makedirs(output_folder_path, exist_ok=True)

tracker.add_step("Configuration", "success", f"Client: {client_name}, Date: {base_date_str}, Category: {product_category}",
                {"depots_path": depots_path, "output_path": output_folder_path, "sheet_name": sheet_name})

print("✅ Initialization complete!")
print(f"   Client: {client_name}")
print(f"   Planning Date: {base_date_str}")
print(f"   Product Category: {product_category}")

# Initialize status tracker (after modules are loaded)
from module_status_tracker import get_tracker, reset_tracker
tracker = get_tracker()
reset_tracker()  # Reset for fresh run
tracker.start()

tracker.add_step("Drive Mount", "success", drive_mount_message)
tracker.add_step("Module Loading", "success", "All modules loaded and reloaded")

# Load parameters
from google.colab import userdata
try:
    api_key = userdata.get('apiKey_HERE')
    if not api_key:
        raise ValueError("API key not found in Colab secrets")
    tracker.add_step("API Key", "success", "API key loaded from Colab secrets")
except Exception as e:
    tracker.add_step("API Key", "error", f"Failed to load API key: {str(e)}",
                    {"hint": "Make sure 'apiKey_HERE' is set in Colab Secrets (🔑 icon)"})
    raise

# Create output folder if it doesn't exist
os.makedirs(output_folder_path, exist_ok=True)

tracker.add_step("Configuration", "success", f"Client: {client_name}, Date: {base_date_str}, Category: {product_category}",
                {"depots_path": depots_path, "output_path": output_folder_path, "sheet_name": sheet_name})

print("✅ Initialization complete!")
print(f"   Client: {client_name}")
print(f"   Planning Date: {base_date_str}")
print(f"   Product Category: {product_category}")

✅ Google Drive is already mounted
✅ All modules loaded successfully
✅ Initialization complete!
   Client: Wigger
   Planning Date: 2026-02-12
   Product Category: Ohne Modifikation
✅ Initialization complete!
   Client: Wigger
   Planning Date: 2026-02-12
   Product Category: Ohne Modifikation


In [4]:
# @title Step 2: Upload and Validate Input File

print("="*80)
print("STEP 2: FILE UPLOAD AND VALIDATION")
print("="*80)

# Upload file
try:
    df_raw = module_upload.upload_excel_via_widget(sheet_name)
    tracker.add_step("File Upload", "success", f"File uploaded successfully",
                    {"shape": f"{df_raw.shape[0]} rows × {df_raw.shape[1]} columns"})
except Exception as e:
    tracker.add_step("File Upload", "error", f"Failed to upload file: {str(e)}",
                    {"hint": "Make sure you select a valid Excel file (.xlsx)"})
    raise

# Validate input file
from module_input_validation import validate_input_file
is_valid, errors, warnings = validate_input_file(df_raw, client_name, sheet_name)

if not is_valid:
    print("\n❌ INPUT FILE VALIDATION FAILED:")
    print("-"*80)
    for error in errors:
        print(error)
    tracker.add_step("Input Validation", "error", "Input file validation failed", {"errors": errors})
    raise ValueError("Input file validation failed. Please fix the errors above and try again.")
else:
    tracker.add_step("Input Validation", "success", "Input file validated successfully")
    if warnings:
        print("\n⚠️  WARNINGS (non-critical):")
        print("-"*80)
        for warning in warnings:
            print(warning)
        tracker.add_step("Input Validation", "warning", "Input file has warnings", {"warnings": warnings})

print(f"\n✅ File loaded: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
print("="*80)

STEP 2: FILE UPLOAD AND VALIDATION


Saving Wigger - Auftragsspeicher - Touren (27).xlsx to Wigger - Auftragsspeicher - Touren (27).xlsx
📁 Uploaded file: Wigger - Auftragsspeicher - Touren (27).xlsx
📄 Loading sheet: 'Touren - BEX'

✅ File loaded: 170 rows, 33 columns


In [5]:
# @title Step 3: Data Cleaning and Filtering

print("="*80)
print("STEP 3: DATA CLEANING AND FILTERING")
print("="*80)

# Validate depots file first
from module_input_validation import InputValidator
validator = InputValidator(client_name)
depots_valid, depots_errors, depots_warnings = validator.validate_depots_file(depots_path)

if not depots_valid:
    print("\n❌ DEPOTS FILE VALIDATION FAILED:")
    print("-"*80)
    for error in depots_errors:
        print(error)
    tracker.add_step("Depots Validation", "error", "Depots file validation failed", {"errors": depots_errors})
    raise ValueError("Depots file validation failed. Please fix the errors above.")
else:
    tracker.add_step("Depots Validation", "success", "Depots file validated")
    if depots_warnings:
        for warning in depots_warnings:
            print(f"⚠️  {warning}")
        tracker.add_step("Depots Validation", "warning", "Depots file has warnings", {"warnings": depots_warnings})

# Clean data
try:
    df_clean = module_clean.clean_and_process_data(df_raw, base_date_str, output_folder_path, client_name)

    # Validate after cleaning
    clean_valid, clean_errors, clean_warnings = validator.validate_after_cleaning(df_clean, df_raw)

    if not clean_valid:
        print("\n❌ DATA CLEANING VALIDATION FAILED:")
        print("-"*80)
        for error in clean_errors:
            print(error)
        tracker.add_step("Data Cleaning", "error", "Data cleaning validation failed", {"errors": clean_errors})
        raise ValueError("Data cleaning validation failed. Please check your input file.")
    else:
        tracker.add_step("Data Cleaning", "success",
                        f"Data cleaned: {len(df_clean)} rows remaining (from {len(df_raw)} original)",
                        {"rows_before": len(df_raw), "rows_after": len(df_clean)})
        if clean_warnings:
            print("\n⚠️  CLEANING WARNINGS:")
            print("-"*80)
            for warning in clean_warnings:
                print(warning)
            tracker.add_step("Data Cleaning", "warning", "Data cleaning has warnings", {"warnings": clean_warnings})

    print(f"\n✅ Data cleaned successfully")
    print(f"   Original rows: {len(df_raw)}")
    print(f"   Cleaned rows: {len(df_clean)}")
    print(f"   Output folder: {output_folder_path}")
    print("="*80)

except Exception as e:
    tracker.add_step("Data Cleaning", "error", f"Data cleaning failed: {str(e)}",
                    {"hint": "Check that date format matches 'dd.mm.yy HH:MM' and branch clusters are valid"})
    raise

STEP 3: DATA CLEANING AND FILTERING
Folder created at /content/drive/MyDrive/Colab Notebooks/Wigger/outputs/2026-02-12

⚠️  CLEANING WARNINGS:
--------------------------------------------------------------------------------
⚠️  167 rows were filtered out during cleaning
   → 3 rows remain for processing

✅ Data cleaned successfully
   Original rows: 170
   Cleaned rows: 3
   Output folder: /content/drive/MyDrive/Colab Notebooks/Wigger/outputs/2026-02-12


In [6]:
# @title Step 4: Geocoding Addresses

print("="*80)
print("STEP 4: GEOCODING ADDRESSES")
print("="*80)
print("This may take a few minutes depending on the number of addresses...")

try:
    df_geocoded = module_geocoding.geocoding_cleaned_data(df_clean, api_key, client_name)

    # Validate geocoding results
    geocoding_valid, geocoding_errors, geocoding_warnings = validator.validate_geocoding(df_geocoded)

    if not geocoding_valid:
        print("\n❌ GEOCODING VALIDATION FAILED:")
        print("-"*80)
        for error in geocoding_errors:
            print(error)
        tracker.add_step("Geocoding", "error", "Geocoding validation failed", {"errors": geocoding_errors})
        raise ValueError("Geocoding validation failed. Please check address data.")
    else:
        tracker.add_step("Geocoding", "success",
                        f"Geocoding completed: {df_geocoded.shape[0]} addresses processed",
                        {"shape": f"{df_geocoded.shape[0]} rows × {df_geocoded.shape[1]} columns"})
        if geocoding_warnings:
            print("\n⚠️  GEOCODING WARNINGS:")
            print("-"*80)
            for warning in geocoding_warnings:
                print(warning)
            tracker.add_step("Geocoding", "warning", "Geocoding has warnings", {"warnings": geocoding_warnings})

    print(f"\n✅ Geocoding completed successfully")
    print(f"   Geocoded data shape: {df_geocoded.shape[0]} rows × {df_geocoded.shape[1]} columns")
    print("="*80)

except Exception as e:
    tracker.add_step("Geocoding", "error", f"Geocoding failed: {str(e)}",
                    {"hint": "Check API key is valid and address columns contain valid data"})
    raise

STEP 4: GEOCODING ADDRESSES
This may take a few minutes depending on the number of addresses...

✅ Geocoding completed successfully
   Geocoded data shape: 3 rows × 61 columns


In [7]:
# @title Step 5: Vehicle Routing Planning (VRP)

print("="*80)
print("STEP 5: VEHICLE ROUTING PLANNING")
print("="*80)
print("Creating VRP problem definition...")

import math

def replace_nan_with_none_recursive(obj):
    """Recursively replaces NaN and Inf float values in a nested dict/list with None."""
    if isinstance(obj, dict):
        return {k: replace_nan_with_none_recursive(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [replace_nan_with_none_recursive(elem) for elem in obj]
    elif isinstance(obj, float) and (math.isnan(obj) or math.isinf(obj)):
        return None
    else:
        return obj

# Create VRP problem definition
try:
    vrp_problem_statement_json = module_vrp.vrp_problem_definition(
        df_geocoded, api_key, depots_path, base_date_str, product_category, client_name
    )

    # Extract info for tracking
    num_jobs = len(vrp_problem_statement_json.get("plan", {}).get("jobs", []))
    num_fleet_types = len(vrp_problem_statement_json.get("fleet", {}).get("types", []))

    tracker.add_step("VRP Problem Definition", "success",
                    f"VRP problem created: {num_jobs} jobs, {num_fleet_types} fleet types",
                    {"jobs": num_jobs, "fleet_types": num_fleet_types})

except Exception as e:
    tracker.add_step("VRP Problem Definition", "error", f"Failed to create VRP problem: {str(e)}",
                    {"hint": "Check that depots file and geocoded data are valid"})
    raise

# Clean the problem statement JSON before execution
print("Cleaning problem statement...")
vrp_problem_statement_json_cleaned = replace_nan_with_none_recursive(vrp_problem_statement_json)

# Execute VRP
print("Executing VRP optimization (this may take a few minutes)...")
try:
    vrp_response_json = module_vrp.vrp_problem_execution(vrp_problem_statement_json_cleaned, api_key)

    # Check for errors in response
    if "error" in vrp_response_json:
        error_msg = vrp_response_json.get("details", vrp_response_json.get("error", "Unknown error"))
        tracker.add_step("VRP Execution", "error", f"VRP API returned error: {error_msg}",
                       {"status_code": vrp_response_json.get("status_code"),
                        "hint": "Check API key, problem definition, and HERE API status"})
        raise ValueError(f"VRP execution failed: {error_msg}")
    else:
        num_tours = len(vrp_response_json.get("tours", []))
        num_unassigned = len(vrp_response_json.get("unassigned", []))
        tracker.add_step("VRP Execution", "success",
                        f"VRP optimization completed: {num_tours} tours created, {num_unassigned} unassigned jobs",
                        {"tours": num_tours, "unassigned": num_unassigned})

        if num_unassigned > 0:
            print(f"\n⚠️  {num_unassigned} jobs could not be assigned to tours")
            print("   Check the unassigned jobs section in the results for details")

    print(f"\n✅ VRP optimization completed")
    print(f"   Tours created: {num_tours}")
    print(f"   Unassigned jobs: {num_unassigned}")
    print("="*80)

except Exception as e:
    tracker.add_step("VRP Execution", "error", f"VRP execution failed: {str(e)}",
                    {"hint": "Check API key, network connection, and HERE API service status"})
    raise


STEP 5: VEHICLE ROUTING PLANNING
Creating VRP problem definition...
Depots DataFrame loaded with 1 rows.
Filtered depots DataFrame with 1 rows.
Generated 2 fleet types.
Generated 3 jobs.
Dynamic JSON object created.
Cleaning problem statement...
Executing VRP optimization (this may take a few minutes)...
Request for VRP_Execution to HERE was successful.

✅ VRP optimization completed
   Tours created: 1
   Unassigned jobs: 0


In [8]:
# @title Step 6: Merge Results and Export

print("="*80)
print("STEP 6: MERGE RESULTS AND EXPORT")
print("="*80)

try:
    # Merge results
    merged_df, df_unassigned_jobs = module_results.merge_results(
        vrp_response_json, df_geocoded, output_folder_path, depots_path
    )

    tracker.add_step("Results Merging", "success",
                    f"Results merged: {merged_df.shape[0]} activities",
                    {"shape": f"{merged_df.shape[0]} rows × {merged_df.shape[1]} columns",
                     "unassigned_jobs": len(df_unassigned_jobs)})

    # Save main results CSV
    main_csv_path = output_folder_path + '/' + base_date_str + '.csv'
    merged_df.to_csv(main_csv_path)
    tracker.add_step("Main CSV Export", "success", f"Main results saved to CSV", {"path": main_csv_path})

    # Create bexOS export
    merged_df_bexOS = module_export.create_bexos_import_csv(merged_df)

    # Save bexOS export
    bexos_csv_path = output_folder_path + '/' + base_date_str + '_import_to_bexOS.csv'
    merged_df_bexOS.to_csv(bexos_csv_path, index=False, sep=",", encoding="utf-8")
    tracker.add_step("bexOS CSV Export", "success", f"bexOS import file saved", {"path": bexos_csv_path})

    print(f"\n✅ Results exported successfully")
    print(f"   Main CSV: {main_csv_path}")
    print(f"   bexOS CSV: {bexos_csv_path}")
    print(f"   Unassigned jobs: {len(df_unassigned_jobs)}")
    print("="*80)

except Exception as e:
    tracker.add_step("Results Export", "error", f"Failed to export results: {str(e)}",
                    {"hint": "Check that output folder exists and has write permissions"})
    raise

STEP 6: MERGE RESULTS AND EXPORT
Depots DataFrame loaded with 1 rows.

✅ Results exported successfully
   Main CSV: /content/drive/MyDrive/Colab Notebooks/Wigger/outputs/2026-02-12/2026-02-12.csv
   bexOS CSV: /content/drive/MyDrive/Colab Notebooks/Wigger/outputs/2026-02-12/2026-02-12_import_to_bexOS.csv
   Unassigned jobs: 0


In [9]:
# @title Step 7: Create Visualization Map

print("="*80)
print("STEP 7: CREATE VISUALIZATION MAP")
print("="*80)

try:
    map = module_map.create_map(merged_df, df_geocoded, df_unassigned_jobs, base_date_str)

    # Save the map
    map_path = output_folder_path + '/' + base_date_str + '_tours_map.html'
    map.save(map_path)

    # Display map in Colab
    print(f"\n✅ Map created successfully")
    print(f"   Map file: {map_path}")
    if len(df_unassigned_jobs) > 0:
        print(f"   ⚠️  {len(df_unassigned_jobs)} unassigned orders are shown on the map")
    print("\nDisplaying interactive map...")
    module_map.display_map_in_colab(map, width=1000, height=600)

    tracker.add_step("Map Creation", "success", f"Interactive map created and saved",
                    {"path": map_path, "unassigned_jobs": len(df_unassigned_jobs)})

    print("="*80)

except Exception as e:
    tracker.add_step("Map Creation", "error", f"Failed to create map: {str(e)}",
                    {"hint": "Check that merged data contains valid location coordinates"})
    raise

STEP 7: CREATE VISUALIZATION MAP

✅ Map created successfully
   Map file: /content/drive/MyDrive/Colab Notebooks/Wigger/outputs/2026-02-12/2026-02-12_tours_map.html

Displaying interactive map...


In [10]:
# @title Step 8: Tour Planning Summary and Map Display

print("="*80)
print("STEP 8: TOUR PLANNING SUMMARY")
print("="*80)

try:
    # Display tour summary with statistics
    summary_df = module_tour_summary.display_tour_summary(
        vrp_response_json,
        merged_df,
        df_unassigned_jobs,
        base_date_str,
        display_in_colab=True
    )

    tracker.add_step("Tour Summary", "success", f"Tour summary displayed",
                    {"tours": len(vrp_response_json.get('tours', [])),
                     "unassigned_jobs": len(df_unassigned_jobs)})

    print("="*80)

except Exception as e:
    tracker.add_step("Tour Summary", "error", f"Failed to display tour summary: {str(e)}",
                    {"hint": "Check that VRP response and merged data are available"})
    print(f"⚠️  Error displaying tour summary: {str(e)}")
    print("="*80)

STEP 8: TOUR PLANNING SUMMARY
TOUR PLANNING SUMMARY
Planning Date: 2026-02-12

Tour Statistics:
--------------------------------------------------------------------------------
              Vehicle ID                Type ID  Stops  Jobs  Pickups  Deliveries  Duration (hours)  Distance (km)           Start Time             End Time
Wigger-500-LKW_7_5-450_1 Wigger-500-LKW_7_5-450      6     5        3           3              2.79          81.24 2026-02-12T08:00:00Z 2026-02-12T10:47:41Z
                   TOTAL                      -      6     5        3           3              2.79          81.24                    -                    -

--------------------------------------------------------------------------------
Unassigned Jobs: 0


In [11]:
# @title Final Status Summary

print("\n" + "="*80)
print("FINAL PROCESS SUMMARY")
print("="*80)

# Check if tracker is defined (in case initialization failed)
if 'tracker' not in globals():
    print("⚠️  Status tracker not initialized. Please run Step 1 (Initialization) first.")
    print("="*80)
else:
    # Mark process as complete
    tracker.end()

    # Print comprehensive summary
    tracker.print_summary()

    # Display steps as DataFrame
    print("\nDETAILED STEP LOG:")
    print("-"*80)
    steps_df = tracker.get_steps_dataframe()
    if not steps_df.empty:
        display(steps_df[['step', 'status', 'message']])

    # Final status
    summary = tracker.get_summary()
    if summary['status'] == 'success':
        print("\n🎉 PROCESS COMPLETED SUCCESSFULLY!")
        print(f"   All {summary['successful_steps']} steps completed")
        if 'output_folder_path' in globals():
            print(f"   Output files saved to: {output_folder_path}")
    elif summary['status'] == 'error':
        print("\n❌ PROCESS COMPLETED WITH ERRORS")
        print(f"   {summary['errors']} error(s) occurred - please review and fix")
        print("   Most errors come from wrong input files - check the validation messages above")
    else:
        print("\n⚠️  PROCESS COMPLETED WITH WARNINGS")
        print(f"   {summary['warnings']} warning(s) - review for potential issues")

    print("="*80)



FINAL PROCESS SUMMARY

TOUR PLANNING PROCESS SUMMARY
Total Steps: 17
Successful: 16
Errors: 0
Warnings: 1
Duration: 18.86 seconds

Overall Status: WARNING

STEP DETAILS:
--------------------------------------------------------------------------------
1. [✓] Drive Mount: Google Drive was already mounted
2. [✓] Module Loading: All modules loaded and reloaded
3. [✓] API Key: API key loaded from Colab secrets
4. [✓] Configuration: Client: Wigger, Date: 2026-02-12, Category: Ohne Modifikation
   - depots_path: /content/drive/MyDrive/Colab Notebooks/Wigger/depots/Depots_geocoded.xlsx
   - output_path: /content/drive/MyDrive/Colab Notebooks/Wigger/outputs/2026-02-12
   - sheet_name: Touren - BEX
5. [✓] File Upload: File uploaded successfully
   - shape: 170 rows × 33 columns
6. [✓] Input Validation: Input file validated successfully
7. [✓] Depots Validation: Depots file validated
8. [✓] Data Cleaning: Data cleaned: 3 rows remaining (from 170 original)
   - rows_before: 170
   - rows_after: 3

,step,status,message
0,Drive Mount,success,Google Drive was...
1,Module Loading,success,All modules load...
2,API Key,success,API key loaded f...
3,Configuration,success,"Client: Wigger, ..."
4,File Upload,success,File uploaded su...
5,Input Validation,success,Input file valid...
6,Depots Validation,success,Depots file vali...
7,Data Cleaning,success,Data cleaned: 3 ...
8,Data Cleaning,warning,Data cleaning ha...
9,Geocoding,success,Geocoding comple...



⚠️  PROCESS COMPLETED WITH WARNINGS
   1 warning(s) - review for potential issues
